# 👖 Variational Autoencoders - Fashion-MNIST

In this notebook, we'll walk through the steps required to train your own autoencoder on the fashion MNIST dataset.

The code has been adapted from the excellent [VAE tutorial](https://keras.io/examples/generative/vae/) created by Francois Chollet, available on the Keras website.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
os.environ["KERAS_BACKEND"] = "torch"  # Keras only loads the dataset; this keeps TensorFlow out of the PyTorch process
from keras import datasets
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

from scipy.stats import norm

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display

# Lightning's Trainer picks the best available device automatically
# (accelerator="auto"), so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 100
VALIDATION_SPLIT = 0.2
EMBEDDING_DIM = 2
EPOCHS = 5
BETA = 500

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data - Keras downloads it once (cached in ~/.keras/datasets)
# and returns NumPy arrays: images (N, 28, 28) uint8, labels (N,)
(x_train, y_train), (x_test, y_test) = datasets.fashion_mnist.load_data()

# Convert to PyTorch tensors
(x_train, y_train) = (torch.from_numpy(x_train), torch.from_numpy(y_train).long())
(x_test, y_test) = (torch.from_numpy(x_test), torch.from_numpy(y_test).long())

In [ ]:
# Preprocess the data


def preprocess(imgs):
    """
    Normalize and reshape the images
    """
    imgs = imgs.float() / 255.0
    imgs = F.pad(imgs, (2, 2, 2, 2), value=0.0)  # 28x28 -> 32x32
    imgs = imgs.unsqueeze(1)  # add the channel axis: PyTorch is NCHW -> (N, 1, 32, 32)
    return imgs


x_train = preprocess(x_train)
x_test = preprocess(x_test)

In [ ]:
# Show some items of clothing from the training set
display(x_train)

## 2. Build the variational autoencoder <a name="build"></a>

In [ ]:
class Sampling(nn.Module):
    """The reparameterization trick: z = mean + std * epsilon, epsilon ~ N(0, I)."""

    def forward(self, z_mean, z_log_var):
        epsilon = torch.randn_like(z_mean)
        return z_mean + torch.exp(0.5 * z_log_var) * epsilon

In [ ]:
# Encoder
# padding=1 with a 3x3 kernel plays the role of Keras' padding="same"
# (each stride-2 conv halves the image size: 32 -> 16 -> 8 -> 4)
shape_before_flattening = (128, IMAGE_SIZE // 8, IMAGE_SIZE // 8)  # the decoder will need this!


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.z_mean = nn.Linear(int(np.prod(shape_before_flattening)), EMBEDDING_DIM)
        self.z_log_var = nn.Linear(int(np.prod(shape_before_flattening)), EMBEDDING_DIM)
        self.sampling = Sampling()

    def forward(self, x):
        x = self.conv(x)
        z_mean = self.z_mean(x)
        z_log_var = self.z_log_var(x)
        z = self.sampling(z_mean, z_log_var)
        return z_mean, z_log_var, z


encoder = Encoder()
print(encoder)
print(f"Total params: {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# Decoder
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(EMBEDDING_DIM, int(np.prod(shape_before_flattening)))
        # stride=2, padding=1, output_padding=1 doubles the size exactly,
        # like Keras' Conv2DTranspose(strides=2, padding="same"): 4 -> 8 -> 16 -> 32
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 128, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),  # pixel values in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, *shape_before_flattening)  # Reshape
        return self.deconv(x)


decoder = Decoder()
print(decoder)
print(f"Total params: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
class VAE(L.LightningModule):
    def __init__(self, encoder, decoder, learning_rate=0.0005):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.learning_rate = learning_rate

    def forward(self, inputs):
        """Call the model on a particular input."""
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return z_mean, z_log_var, reconstruction

    def compute_losses(self, data):
        z_mean, z_log_var, reconstruction = self(data)
        # Mean binary cross-entropy over all pixels (and the batch), scaled by BETA
        reconstruction_loss = BETA * F.binary_cross_entropy(reconstruction, data)
        # KL divergence to N(0, I): summed over latent dims, averaged over the batch
        kl_loss = torch.mean(
            torch.sum(-0.5 * (1 + z_log_var - z_mean.pow(2) - z_log_var.exp()), dim=1)
        )
        total_loss = reconstruction_loss + kl_loss
        return total_loss, reconstruction_loss, kl_loss

    def _log_losses(self, losses, prefix):
        names = ["loss", "reconstruction_loss", "kl_loss"]
        for name, value in zip(names, losses):
            # on_epoch=True averages over the epoch, like Keras' metrics.Mean trackers
            self.log(prefix + name, value, on_step=False, on_epoch=True, prog_bar=True)

    def training_step(self, data, batch_idx):
        """Step run during training (Lightning does backward + optimizer step)."""
        losses = self.compute_losses(data)
        self._log_losses(losses, "")
        return losses[0]  # the total loss is what gets minimised

    def validation_step(self, data, batch_idx):
        """Step run during validation."""
        self._log_losses(self.compute_losses(data), "val_")

    def predict_step(self, data, batch_idx):
        return self(data)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [ ]:
# Create a variational autoencoder
vae = VAE(encoder, decoder)

## 3. Train the variational autoencoder <a name="train"></a>

In [ ]:
# "Compile": Lightning has no compile(); the optimizer is built in
# configure_optimizers() from this learning rate when training starts
vae.learning_rate = 0.0005

In [ ]:
# Create a model save checkpoint (keeps the epoch with the lowest training loss)
model_checkpoint_callback = ModelCheckpoint(
    dirpath="./checkpoint",
    filename="vae",
    monitor="loss",
    mode="min",
    save_top_k=1,
    save_on_train_epoch_end=True,
)
tensorboard_logger = TensorBoardLogger(save_dir="./logs")

In [ ]:
train_loader = DataLoader(x_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(x_test, batch_size=BATCH_SIZE)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[model_checkpoint_callback],
    logger=tensorboard_logger,
)
trainer.fit(vae, train_loader, val_loader)

In [ ]:
# Save the final models
trainer.save_checkpoint("./models/vae.ckpt")  # full Lightning checkpoint
torch.save(encoder.state_dict(), "./models/encoder.pt")  # plain PyTorch weights
torch.save(decoder.state_dict(), "./models/decoder.pt")

## 3. Reconstruct using the variational autoencoder <a name="reconstruct"></a>

In [ ]:
# Select a subset of the test set
n_to_predict = 5000
example_images = x_test[:n_to_predict]
example_labels = y_test[:n_to_predict]

In [ ]:
# Create autoencoder predictions and display
# trainer.predict runs predict_step over a DataLoader (in eval mode, no gradients)
predictions = trainer.predict(vae, DataLoader(example_images, batch_size=BATCH_SIZE))
z_mean, z_log_var, reconstructions = (torch.cat(t) for t in zip(*predictions))
print("Example real clothing items")
display(example_images)
print("Reconstructions")
display(reconstructions)

## 4. Embed using the encoder <a name="encode"></a>

In [ ]:
# Encode the example images
# encoder is a plain nn.Module, so we can call it directly
# (after training, Lightning has moved the model back to the CPU)
vae.eval()
with torch.no_grad():
    z_mean, z_var, z = encoder(example_images)
z = z.numpy()  # NumPy for plotting

In [ ]:
# Some examples of the embeddings
print(z[:10])

In [ ]:
# Show the encoded points in 2D space
figsize = 8

plt.figure(figsize=(figsize, figsize))
plt.scatter(z[:, 0], z[:, 1], c="black", alpha=0.5, s=3)
plt.show()

## 5. Generate using the decoder <a name="decode"></a>

In [ ]:
# Sample some points in the latent space, from the standard normal distribution
grid_width, grid_height = (6, 3)
z_sample = np.random.normal(size=(grid_width * grid_height, 2))

In [ ]:
# Decode the sampled points
with torch.no_grad():
    reconstructions = decoder(torch.tensor(z_sample, dtype=torch.float32))
reconstructions = reconstructions[:, 0].numpy()  # (N, 1, 32, 32) -> (N, 32, 32)

In [ ]:
# Convert original embeddings and sampled embeddings to p-values
p = norm.cdf(z)
p_sample = norm.cdf(z_sample)

In [ ]:
# Draw a plot of...
figsize = 8
plt.figure(figsize=(figsize, figsize))

# ... the original embeddings ...
plt.scatter(z[:, 0], z[:, 1], c="black", alpha=0.5, s=2)

# ... and the newly generated points in the latent space
plt.scatter(z_sample[:, 0], z_sample[:, 1], c="#00B0F0", alpha=1, s=40)
plt.show()

# Add underneath a grid of the decoded images
fig = plt.figure(figsize=(figsize, grid_height * 2))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        str(np.round(z_sample[i, :], 1)),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(reconstructions[i, :, :], cmap="Greys")

## 6. Explore the latent space <a name="explore"></a>

In [ ]:
# Colour the embeddings by their label (clothing type - see table)
figsize = 8
fig = plt.figure(figsize=(figsize * 2, figsize))
ax = fig.add_subplot(1, 2, 1)
plot_1 = ax.scatter(
    z[:, 0], z[:, 1], cmap="rainbow", c=example_labels, alpha=0.8, s=3
)
plt.colorbar(plot_1)
ax = fig.add_subplot(1, 2, 2)
plot_2 = ax.scatter(
    p[:, 0], p[:, 1], cmap="rainbow", c=example_labels, alpha=0.8, s=3
)
plt.show()

| ID | Clothing Label |
| :- | :- |
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

In [ ]:
# Colour the embeddings by their label (clothing type - see table)
figsize = 12
grid_size = 15
plt.figure(figsize=(figsize, figsize))
plt.scatter(
    p[:, 0], p[:, 1], cmap="rainbow", c=example_labels, alpha=0.8, s=300
)
plt.colorbar()

x = norm.ppf(np.linspace(0, 1, grid_size))
y = norm.ppf(np.linspace(1, 0, grid_size))
xv, yv = np.meshgrid(x, y)
xv = xv.flatten()
yv = yv.flatten()
grid = np.array(list(zip(xv, yv)))

with torch.no_grad():
    reconstructions = decoder(torch.tensor(grid, dtype=torch.float32))
reconstructions = reconstructions[:, 0].numpy()
# plt.scatter(grid[:, 0], grid[:, 1], c="black", alpha=1, s=10)
plt.show()

fig = plt.figure(figsize=(figsize, figsize))
fig.subplots_adjust(hspace=0.4, wspace=0.4)
for i in range(grid_size**2):
    ax = fig.add_subplot(grid_size, grid_size, i + 1)
    ax.axis("off")
    ax.imshow(reconstructions[i, :, :], cmap="Greys")